In [1]:
import os
import random
import math
from collections import Counter
from pprint import pprint


import numpy as np
import pandas as pd
import networkx as nx
import xml.etree.ElementTree as ET
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score


# torch + pyg
import torch
import torch.nn.functional as F
from torch_geometric.utils import from_networkx, to_undirected, negative_sampling
from torch_geometric.loader import NeighborSampler, NeighborLoader
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GAE, VGAE

import networkx as nx
import pandas as pd
import xml.etree.ElementTree as ET

/home/ryanm/miniconda3/envs/graphtorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#helper functions

def parse_graphml(file_path):
    """Parse GraphML file and return NetworkX graph"""
    try:
        G = nx.read_graphml(file_path)
        return G
    except Exception:
        return parse_graphml_manual(file_path)

def parse_graphml_manual(file_path):
    """Manual fallback parsing of GraphML"""
    tree = ET.parse(file_path)
    root = tree.getroot()
    G = nx.Graph()

    # Parse keys
    keys = {}
    for key_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}key'):
        keys[key_elem.get('id')] = key_elem.get('attr.name')

    # Parse nodes
    for node_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}node'):
        G.add_node(node_elem.get('id'))

    # Parse edges with attributes
    for edge_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}edge'):
        source = edge_elem.get('source')
        target = edge_elem.get('target')
        edge_attrs = {}

        for data_elem in edge_elem.findall('.//{http://graphml.graphdrawing.org/xmlns}data'):
            key_id = data_elem.get('key')
            attr_name = keys.get(key_id)
            if attr_name:
                try:
                    edge_attrs[attr_name] = float(data_elem.text)
                except:
                    edge_attrs[attr_name] = data_elem.text

        G.add_edge(source, target, **edge_attrs)

    return G

def extract_edge_features(G):
    """Extract edge features as pandas DataFrame"""
    rows = []
    for u, v, data in G.edges(data=True):
        row = {"source": u, "target": v}
        row.update(data)
        rows.append(row)
    return pd.DataFrame(rows)


In [4]:
file_path = "data/0.1M-Stratified-Multi.graphml"

# Load graph
G = parse_graphml(file_path)

print("Number of nodes :", G.number_of_nodes())
print("Number of edges :", G.number_of_edges())

# Extract edge features
edge_df = extract_edge_features(G)

print("\nEdge feature columns :", edge_df.columns.tolist())
print("\nEdge features shape :", edge_df.shape)

print("\nSummary statistics:\n")
print(edge_df.describe())

Number of nodes : 41073
Number of edges : 100000

Edge feature columns : ['source', 'target', 'TotPkts', 'TotBytes', 'SrcBytes', 'Dur', 'Proto_encoded', 'Dir_encoded', 'State_encoded', 'ActivityLabel']

Edge features shape : (100000, 10)

Summary statistics:

             TotPkts       TotBytes       SrcBytes            Dur  \
count  100000.000000  100000.000000  100000.000000  100000.000000   
mean        0.000161      -0.000150       0.000881       0.003185   
std         0.612019       0.745613       0.810429       1.003557   
min        -0.011045      -0.012529      -0.013294      -0.373111   
25%        -0.010884      -0.012494      -0.013254      -0.373111   
50%        -0.010884      -0.012480      -0.013250      -0.373110   
75%        -0.010081      -0.012323      -0.013044      -0.364371   
max       149.951002     183.902708      72.277354       3.705196   

       Proto_encoded    Dir_encoded  State_encoded  ActivityLabel  
count  100000.000000  100000.000000  100000.000000